To fit run di following notebooks, if you never do am yet, you gats deploy model wey dey use `text-embedding-ada-002` as base model and put di deployment name inside .env file as `AZURE_OPENAI_EMBEDDINGS_ENDPOINT`


In [ ]:
import os
import pandas as pd
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],  # this is also the default, it can be omitted
  api_version = "2024-10-21",
  azure_endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
  )

model = os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "../embedding_index_3m.json"


Neext, we go load di Embedding Index inside one Pandas Dataframe. Di Embedding Index dey store for one JSON file wey dem dey call `embedding_index_3m.json`. Di Embedding Index get di Embeddings for each YouTube transcripts dem reach late Oct 2023.


In [ ]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

Next, we go create one function wey dem go call `get_videos` wey go search the Embedding Index for the query. The function go return top 5 videos wey be like the query pass. The function dey work like dis:

1. First, dem go make one copy of the Embedding Index.
2. Next, dem go calculate the Embedding for the query use OpenAI Embedding API.
3. After dat, dem go create one new column for the Embedding Index wey dem go call `similarity`. The `similarity` column get the cosine similarity between the query Embedding and the Embedding for each video segment.
4. Next, dem go filter the Embedding Index by the `similarity` column. The Embedding Index go only carry videos wey get cosine similarity wey pass or equal to 0.75.
5. Last last, dem go arrange the Embedding Index by the `similarity` column and return the top 5 videos.


In [ ]:
def cosine_similarity(a, b):
    if len(a) > len(b):
        b = np.pad(b, (0, len(a) - len(b)), 'constant')
    elif len(b) > len(a):
        a = np.pad(a, (0, len(b) - len(a)), 'constant')
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding

    # create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

Dis function easy die, e just dey print out di results wey search query find.


In [ ]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

1. Fus, dem go load the Embedding Index for inside Pandas Dataframe.
2. Next, dem go ask di user to put for inside query.
3. Den di `get_videos` function go run to check di Embedding Index for di query.
4. Finally, di `display_results` function go run to show di results to di user.
5. Di user go still asked to put another query. Dis kain process go dey continue till di user put `exit`.

![](../../../../translated_images/pcm/notebook-search.1e320b9c7fcbb0bc.webp)

Dem go ask you to put query. Put any query and press enter. Di app go bring list of videos wey get connection with di query. Di app go still bring link to di part for inside di video wey dey contain di answer to di question.

Dis na some queries wey you fit try:

- Wetin be Azure Machine Learning?
- How convolutional neural networks dey work?
- Wetin be neural network?
- I fit use Jupyter Notebooks with Azure Machine Learning?
- Wetin be ONNX?


In [ ]:
pd_vectors = load_dataset(DATASET_NAME)

# get user query from input
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
